In [7]:
from pyspark import SparkConf, SparkContext
import re

conf = (SparkConf()
        .setAppName("LR_4_Analyze_web_server_logs")
        .setMaster("local[*]")
)
sc = SparkContext(conf=conf)
logs_rdd = sc.textFile("logfiles.log")

# Функция для парсинга строки лога и извлечения нужных данных
def parse_log(log_line):
    pattern = r'(\S+) - - \[(.*?)\] "(.*?)" (\d{3}) (\d+)'
    match = re.match(pattern, log_line)
    if match:
        ip = match.group(1)
        request = match.group(3)
        status_code = int(match.group(4))
        response_size = int(match.group(5))
        return (ip, request, status_code, response_size)
    return None

# Применим функцию для парсинга
parsed_logs_rdd = logs_rdd.map(parse_log).filter(lambda x: x is not None)
parsed_logs_rdd.


### Общая статистика по логам
- Посчитайте общее количество запросов.
- Рассчитайте средний размер ответа сервера (в байтах) по всем запросам.
- Определите количество уникальных IP-адресов, которые обращались к серверу.

In [10]:
# Общее кол-во запросов
print(f"Общее кол-во запросов: {parsed_logs_rdd.count()}")

# Средний размер ответа сервера в байтах по всем запросам
sum_answer = parsed_logs_rdd.map(lambda x: x[3]).sum()
print(f"Средний размер ответа сервера (в байтах): {round(sum_answer/parsed_logs_rdd.count())}")

# Количество уникальных IP-адресов
unique_ips = parsed_logs_rdd.map(lambda x: x[0]).distinct().count()
print(f"Количество уникальных IP-адресов: {unique_ips}")

Общее кол-во запросов: 10000
Средний размер ответа сервера (в байтах): 4999
Количество уникальных IP-адресов: 10000


### Анализ HTTP-статусов
- Посчитайте количество запросов для каждого HTTP-статус-кода (например, сколько 200, сколько 404, сколько 500).
- Определите долю успешных запросов (статус-код 200) от общего числа запросов в процента

In [11]:
# Подсчет количество запросов для каждого HTTP-статус-кода
status_codes = parsed_logs_rdd.map(lambda x: (x[2], 1))
status_codes_count = status_codes.reduceByKey(lambda a, b: a + b)
print("Распределение по HTTP-кодам:")
for code, count in status_codes_count.sortByKey().collect():
    print(f"{code}: {count} запросов")

Распределение по HTTP-кодам:
200: 3850 запросов
301: 780 запросов
302: 773 запросов
400: 734 запросов
401: 781 запросов
403: 735 запросов
404: 820 запросов
500: 797 запросов
502: 730 запросов


In [12]:
# Определение доли успешных запросов (статус-код 200) от общего числа запросов в процента
code_200_cnt = status_codes_count.filter(lambda x: x[0] == 200).collect()[0][1]
total_code = status_codes_count.map(lambda x: x[1]).sum()
print(f"Доля успешных запросов: {code_200_cnt/parsed_logs_rdd.count()*100}%")

Доля успешных запросов: 38.5%


### Анализ эндпоинтов и запросов
- Найдите Топ-5 самых часто запрашиваемых эндпоинтов. Эндпоинт - это часть URL-адреса, которая указывает на конкретный ресурс или функцию на сервере, к которой обращается клиент. Например, в запросе "GET /api/v1/users HTTP/1.0" эндпоинтом является /api/v1/users.
- Посчитайте, сколько запросов каждого типа (GET, POST, PUT и т.д.) было сделано

In [13]:
# Топ-5 самых часто запрашиваемых эндпоинтов
resource_requests = parsed_logs_rdd.map(lambda x: (x[1].split()[1],1)) 
top_resources = resource_requests.reduceByKey(lambda a, b: a + b).top(5, key=lambda x: x[1])
print("Top-5 самых часто запрашиваемых ресурсов:")
for resource, count in top_resources:
    print(f"{resource}: {count} запросов")

Top-5 самых часто запрашиваемых ресурсов:
/search?q=spark: 952 запросов
/docs/api: 949 запросов
/admin/dashboard: 935 запросов
/auth/register: 929 запросов
/: 913 запросов


In [14]:
# Количество запросов каждого типа (GET, POST, PUT и т.д.) 
requests = parsed_logs_rdd.map(lambda x: (x[1].split()[0],1))
requests_count = requests.reduceByKey(lambda a, b: a + b)
print("Распределение по типам запросов:")
for request, count in requests_count.sortByKey().collect():
    print(f"{request}: {count} запросов")

Распределение по типам запросов:
DELETE: 1711 запросов
GET: 1677 запросов
HEAD: 1669 запросов
OPTIONS: 1662 запросов
POST: 1641 запросов
PUT: 1640 запросов


In [15]:
sc.stop()

## Решение DataFrame¶

In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, sum, min, max, length, avg, round, count, substring, btrim, count_distinct, split
from pyspark.sql.types import StructType, StructField, StringType

In [28]:
spark = SparkSession.builder \
        .appName("LR_4_Analyze_web_server_logs") \
        .master("local[*]") \
        .getOrCreate()
#data_schema = 
file_path = "logfiles.log"

df_logs = spark.read.csv(
    file_path,
    header=False,
    inferSchema=True,
    sep=" "
).withColumnsRenamed(
    {"_c0": "ip",
    "_c5": "full_request",
    "_c6": "status_code",
    "_c7": "response_size"}
).select(col("ip"), col("full_request"), col("status_code"), col("response_size"))

df_logs.show(truncate=False)
df_logs.printSchema()

+--------------+--------------------------------+-----------+-------------+
|ip            |full_request                    |status_code|response_size|
+--------------+--------------------------------+-----------+-------------+
|28.225.186.85 |POST /docs/api HTTP/1.0         |200        |4992         |
|180.13.230.30 |DELETE /auth/register HTTP/1.0  |200        |5009         |
|82.67.67.141  |GET /blog/latest HTTP/1.0       |401        |4951         |
|115.106.252.49|PUT /api/v1/products HTTP/1.0   |200        |4921         |
|139.50.169.237|OPTIONS / HTTP/1.0              |500        |4990         |
|201.10.213.158|POST /auth/register HTTP/1.0    |301        |4967         |
|101.109.14.220|POST /docs/api HTTP/1.0         |200        |5043         |
|94.146.162.220|OPTIONS /api/v1/users HTTP/1.0  |301        |5009         |
|211.50.163.201|DELETE /images/logo.png HTTP/1.0|302        |4899         |
|175.48.17.195 |PUT /search?q=spark HTTP/1.0    |302        |5006         |
|199.82.49.8

### Общая статистика по логам
- Посчитайте общее количество запросов.
- Рассчитайте средний размер ответа сервера (в байтах) по всем запросам.
- Определите количество уникальных IP-адресов, которые обращались к серверу.

In [41]:
# Общее кол-во запросов
print(f"Общее кол-во запросов: {df_logs.count()}")

# Средний размер ответа сервера в байтах по всем запросам
print(f"Средний размер ответа сервера по всем запросам: {df_logs.agg(avg(col("response_size"))).collect()[0][0]}")

# Количество уникальных IP-адресов
print(f"Количество уникальных IP-адресов: {df_logs.select(col("ip")).distinct().count()}")

Общее кол-во запросов: 10000
Средний размер ответа сервера по всем запросам: 4999.0504
Количество уникальных IP-адресов: 10000


### Анализ HTTP-статусов
- Посчитайте количество запросов для каждого HTTP-статус-кода (например, сколько 200, сколько 404, сколько 500).
- Определите долю успешных запросов (статус-код 200) от общего числа запросов в процента

In [43]:
df_cnt_request = df_logs.groupBy(col("status_code")).agg(
    count(col("status_code")).alias("cnt_codes")
).orderBy(col("cnt_codes").desc())
print("Кол-во запросов для каждого HTTP сатус кода:")
df_cnt_request.show()

Кол-во запросов для каждого HTTP сатус кода:
+-----------+---------+
|status_code|cnt_codes|
+-----------+---------+
|        200|     3850|
|        404|      820|
|        500|      797|
|        401|      781|
|        301|      780|
|        302|      773|
|        403|      735|
|        400|      734|
|        502|      730|
+-----------+---------+



In [45]:
# Определение доли успешных запросов (статус-код 200) от общего числа запросов в процента
code_200_cnt = df_cnt_request.filter(col("status_code") == 200).collect()[0][1]
print(f"Доля успешных запросов: {code_200_cnt/df_logs.count()*100}%")



Доля успешных запросов: 38.5%


### Анализ эндпоинтов и запросов
- Найдите Топ-5 самых часто запрашиваемых эндпоинтов. Эндпоинт - это часть URL-адреса, которая указывает на конкретный ресурс или функцию на сервере, к которой обращается клиент. Например, в запросе "GET /api/v1/users HTTP/1.0" эндпоинтом является /api/v1/users.
- Посчитайте, сколько запросов каждого типа (GET, POST, PUT и т.д.) было сделано

In [54]:
df_endpoint = df_logs.select(col("full_request"), split(col("full_request"), " ")[1].alias("endpoint")).groupBy(col("endpoint")).agg(
    count(col("endpoint")).alias("cnt_endpoint")
).orderBy(col("cnt_endpoint").desc()).limit(5)
print("ТОП 5 эндпойнтов")
df_endpoint.show()

ТОП 5 эндпойнтов
+----------------+------------+
|        endpoint|cnt_endpoint|
+----------------+------------+
| /search?q=spark|         952|
|       /docs/api|         949|
|/admin/dashboard|         935|
|  /auth/register|         929|
|               /|         913|
+----------------+------------+



In [59]:
df_type_requests = df_logs.select(col("full_request"), split(col("full_request"), " ")[0].alias("type_request")).groupBy(col("type_request")).agg(
    count(col("type_request")).alias("cnt_types")
).orderBy(col("cnt_types").desc())
print("Кол-во запросов каждого типа:")
df_type_requests.show()

Кол-во запросов каждого типа:
+------------+---------+
|type_request|cnt_types|
+------------+---------+
|      DELETE|     1711|
|         GET|     1677|
|        HEAD|     1669|
|     OPTIONS|     1662|
|        POST|     1641|
|         PUT|     1640|
+------------+---------+



In [60]:
spark.stop()